# Hotel Booking Demand: Predicció de cancel·lacions de reserves hoteleres

## 1. Introducció i plantejament del problema

### 1.1. Context i motivació

Les cancel·lacions de reserves representen un problema per a la gestió hotelera, ja que dificulten la previsió de l'ocupació i la planificació dels recursos disponibles.

En aquest projecte estudiarem si és possible predir la cancel·lació d'una reserva hotelera mitjançant tècniques d'aprenentatge computacional.

Per fer-ho, analitzarem un conjunt de dades de reserves hoteleres i compararem diferents models de classificació estudiats a l'assignatura d'Aprenentatge Computacional.

### 1.2. Objectiu del projecte

L'objectiu principal és construir i avaluar models capaços de predir si una reserva hotelera serà cancel·lada.

La pregunta que volem respondre és:

**Podem predir si una reserva serà cancel·lada utilitzant únicament la informació disponible en el moment de fer-la?**

Aquest problema correspon a una tasca de classificació binària, en què la variable objectiu és `is_canceled`:

- **0:** la reserva no ha estat cancel·lada.
- **1:** la reserva ha estat cancel·lada.

### 1.3. Metodologia general

Per resoldre el problema seguirem les etapes següents:

1. Comprensió i exploració del conjunt de dades.
2. Identificació de possibles problemes de qualitat i filtració de dades.
3. Preparació de les dades i definició del protocol de validació.
4. Entrenament i comparació de diferents models de classificació.
5. Selecció i avaluació del model final.
6. Interpretació dels resultats, anàlisi d'errors i conclusions.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (f1_score, average_precision_score, classification_report, confusion_matrix)
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
print('Entorn preparat.')

Entorn preparat.


## 2. Càrrega i comprensió del conjunt de dades

### 2.1. Càrrega de les dades

En primer lloc, carreguem el conjunt de dades *Hotel Booking Demand* mitjançant la llibreria `pandas`.

Comprovarem les dimensions del dataset i visualitzarem les primeres files per obtenir una primera aproximació a la seva estructura.

In [5]:
import pandas as pd

df = pd.read_csv("hotel_bookings.csv")

print("Dimensions del dataset:", df.shape)

display(df.head())


Dimensions del dataset: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### 2.2. Primera observació

El conjunt de dades conté **119.390 registres i 32 atributs**.

Cada fila representa una reserva hotelera, mentre que les columnes recullen informació sobre les característiques de la reserva, el client i el seu estat.

Entre els atributs trobem variables numèriques, com `lead_time`, variables categòriques, com `hotel`, i informació temporal, com `arrival_date_year`.

La variable objectiu és `is_canceled`, que indica si la reserva ha estat cancel·lada.

Abans d'entrenar els models, caldrà analitzar els tipus de dades, els valors perduts i la disponibilitat dels atributs en el moment de fer la predicció.

### 2.3. Anàlisi dels atributs i tipus de dades

Per comprendre millor l'estructura del conjunt de dades, examinarem els noms dels atributs, els seus tipus i el nombre de valors no nuls.

Aquesta anàlisi ens permetrà identificar variables numèriques, categòriques i temporals, així com detectar possibles problemes en la representació de les dades.

In [6]:
# Informació general dels atributs
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

#### Interpretació

El conjunt de dades conté atributs de diferents tipus:

- **Numèrics:** com `lead_time`, `adr` i `stays_in_week_nights`, que representen quantitats.
- **Categòrics:** com `hotel`, `meal` i `market_segment`, que representen categories.
- **Temporals:** com `arrival_date_year`, `arrival_date_month` i `reservation_status_date`, que contenen informació relacionada amb dates.
- **Binaris:** com `is_canceled` i `is_repeated_guest`, que representen dues possibilitats mitjançant els valors 0 i 1.

Cal tenir en compte que el tipus de dada assignat per pandas no sempre coincideix amb la naturalesa de l'atribut. Per exemple, algunes variables numèriques representen categories o identificadors.

També s'observa que algunes columnes tenen valors absents. Analitzarem la seva quantitat i el seu significat en els apartats següents.